# Extração de dados

## 1) Instalação


In [ ]:

!pip -q install -U openai openpyxl pymupdf tenacity tqdm


## 2) Drive e imports


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import re
import json
import time
from pathlib import Path
from datetime import datetime, timezone

import fitz  # PyMuPDF
import pandas as pd
from tqdm.auto import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

from openai import OpenAI, APITimeoutError, APIConnectionError, RateLimitError, APIError

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3) Configuração

In [ ]:
PDF_DIR = "/content/drive/MyDrive/Colab Notebooks/TCC/pdfs_finais"
OUTPUT_CSV = "/content/drive/MyDrive/Colab Notebooks/TCC/extraction_matrix.csv"
OUTPUT_XLSX = "/content/drive/MyDrive/Colab Notebooks/TCC/extraction_matrix.xlsx"
TEXT_DIR = "/content/drive/MyDrive/Colab Notebooks/TCC/extraction_txt"
AUDIT_DIR = "/content/drive/MyDrive/Colab Notebooks/TCC/extraction_audit_json"

MODEL = "gpt-4o"
PROMPT_VERSION = "extraction_js_v2_resume"

REQUEST_PAUSE_SECONDS = 1.0
CHUNK_SIZE = 8000
CHUNK_OVERLAP = 600

# Controle de retomada
REPROCESS_ERRORS = True   # True = tenta novamente os que já estão ERROR
STOP_ON_BILLING_ERROR = True

## 4) API Key


In [ ]:

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY não encontrada em Colab Secrets.")

client = OpenAI(api_key=OPENAI_API_KEY)

## 5) Utilitários


In [ ]:
def ensure_dir(path: str):
    Path(path).mkdir(parents=True, exist_ok=True)

def clean_text(x):
    if x is None:
        return ""
    return str(x).replace("\x00", " ").strip()

def normalize_ws(text: str) -> str:
    text = clean_text(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def list_pdfs(pdf_dir: str):
    return sorted(Path(pdf_dir).rglob("*.pdf"))

def extract_json(text: str):
    text = clean_text(text)
    try:
        return json.loads(text)
    except:
        pass

    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except:
            pass
    return None

def safe_join_list(value):
    if isinstance(value, list):
        return "; ".join([clean_text(v) for v in value if clean_text(v)])
    return clean_text(value)

def chunk_text(full_text: str, chunk_size=12000, overlap=1200):
    text = clean_text(full_text)
    if not text:
        return []

    chunks = []
    start = 0
    n = len(text)

    while start < n:
        end = min(start + chunk_size, n)
        chunks.append(text[start:end])
        if end == n:
            break
        start = max(end - overlap, start + 1)

    return chunks

def is_billing_error(exc: Exception) -> bool:
    msg = str(exc).lower()
    billing_markers = [
        "insufficient_quota",
        "quota",
        "billing",
        "credit",
        "credits",
        "exceeded your current quota"
    ]
    return any(marker in msg for marker in billing_markers)

## 6) Extração do título do PDF


In [ ]:
def extract_title_from_pdf(pdf_path: Path) -> str:
    try:
        doc = fitz.open(str(pdf_path))
        if len(doc) == 0:
            return pdf_path.stem

        page = doc[0]
        page_dict = page.get_text("dict")
        page_height = page.rect.height

        spans = []
        for block in page_dict.get("blocks", []):
            for line in block.get("lines", []):
                line_text = []
                max_size = 0.0
                min_y = None

                for span in line.get("spans", []):
                    txt = clean_text(span.get("text", ""))
                    if not txt:
                        continue
                    size = float(span.get("size", 0.0))
                    bbox = span.get("bbox", [0, 0, 0, 0])
                    y = bbox[1]
                    line_text.append(txt)
                    max_size = max(max_size, size)
                    min_y = y if min_y is None else min(min_y, y)

                joined = normalize_ws(" ".join(line_text))
                if not joined:
                    continue

                low = joined.lower()
                if len(joined) < 8:
                    continue
                if "doi" in low or "copyright" in low:
                    continue
                if "acm" in low and len(joined) < 40:
                    continue
                if "ieee" in low and len(joined) < 40:
                    continue
                if "@" in joined:
                    continue
                if min_y is not None and min_y > page_height * 0.45:
                    continue

                spans.append({
                    "text": joined,
                    "size": max_size,
                    "y": min_y if min_y is not None else 0
                })

        if spans:
            spans = sorted(spans, key=lambda x: (-x["size"], x["y"]))
            top_size = spans[0]["size"]
            candidates = [
                s for s in spans
                if s["size"] >= top_size - 0.8 and s["y"] <= page_height * 0.35
            ]
            if candidates:
                candidates = sorted(candidates, key=lambda x: x["y"])
                title = normalize_ws(" ".join([c["text"] for c in candidates]))
                if len(title) >= 12:
                    return title

        first_page_text = page.get_text("text", sort=True)
        lines = [normalize_ws(l) for l in first_page_text.splitlines() if normalize_ws(l)]
        filtered = []
        for line in lines[:20]:
            low = line.lower()
            if len(line) < 8:
                continue
            if "doi" in low or "copyright" in low or "@" in line:
                continue
            if re.match(r"^\d+$", line):
                continue
            filtered.append(line)

        return filtered[0] if filtered else pdf_path.stem

    except Exception:
        return pdf_path.stem

## 7) Texto completo


In [ ]:
def extract_full_text(pdf_path: Path) -> str:
    doc = fitz.open(str(pdf_path))
    parts = []
    for i, page in enumerate(doc):
        txt = page.get_text("text", sort=True)
        parts.append(f"\n\n--- PAGE {i+1} ---\n{txt}")
    return "".join(parts).strip()

## 8) Seções auxiliares


In [ ]:
SECTION_PATTERNS = {
    "introduction": [r"^(?:\d+|[ivxlcdm]+)?\.?\s*introduction$"],
    "method": [
        r"^(?:\d+|[ivxlcdm]+)?\.?\s*(method|methods|methodology)$",
        r"^(?:\d+|[ivxlcdm]+)?\.?\s*(materials and methods)$",
        r"^(?:\d+|[ivxlcdm]+)?\.?\s*(approach|research method|experimental setup)$"
    ],
    "conclusion": [
        r"^(?:\d+|[ivxlcdm]+)?\.?\s*(conclusion|conclusions)$",
        r"^(?:\d+|[ivxlcdm]+)?\.?\s*(final remarks|concluding remarks)$"
    ]
}

def normalize_heading(line: str) -> str:
    line = clean_text(line).lower()
    line = re.sub(r"[^a-z0-9 ]", "", line)
    line = re.sub(r"\s+", " ", line).strip()
    return line

def split_sections(full_text: str):
    lines = full_text.splitlines()
    found = {}

    for idx, line in enumerate(lines):
        norm = normalize_heading(line)
        for sec, patterns in SECTION_PATTERNS.items():
            for pat in patterns:
                if re.match(pat, norm):
                    if sec not in found:
                        found[sec] = idx

    result = {"introduction": "", "method": "", "conclusion": ""}
    if not found:
        return result, []

    ordered = sorted(found.items(), key=lambda x: x[1])

    for i, (sec, start_idx) in enumerate(ordered):
        end_idx = ordered[i+1][1] if i + 1 < len(ordered) else len(lines)
        result[sec] = "\n".join(lines[start_idx:end_idx]).strip()

    detected = [k for k, v in result.items() if clean_text(v)]
    return result, detected


## 9) Prompts


In [ ]:
def build_chunk_prompt(title: str, chunk_index: int, chunk_text: str) -> str:
    return f"""
You are supporting structured data extraction for a systematic mapping study on JavaScript code smell detection.

At this stage, DO NOT return the final extraction for the whole study.
Your job is only to identify evidence in THIS TEXT CHUNK.

Study title:
{title}

Chunk index:
{chunk_index}

Fields of interest:
- possible tool or approach name
- whether the study explicitly addresses JavaScript
- possible code smell types
- possible detection technique
- possible analysis type
- possible validation type
- possible dataset or projects
- possible limitations
- possible main findings

Rules:
1. Use only this text chunk.
2. Do not infer absent information.
3. Return valid JSON only.

Text chunk:
{chunk_text}

Return exactly this JSON:
{{
  "tool_or_approach_candidates": ["..."],
  "javascript_scope_signals": ["..."],
  "smell_type_signals": ["..."],
  "detection_technique_signals": ["..."],
  "analysis_type_signals": ["..."],
  "validation_type_signals": ["..."],
  "dataset_signals": ["..."],
  "limitation_signals": ["..."],
  "finding_signals": ["..."],
  "evidence_spans": ["quote 1", "quote 2"],
  "observations": "short note"
}}
""".strip()

def build_consolidation_prompt(title: str, sections: dict, chunk_summaries: list) -> str:
    compact_chunks = []
    for ch in chunk_summaries:
        compact_chunks.append({
            "chunk_index": ch["chunk_index"],
            "tool_or_approach_candidates": ch.get("tool_or_approach_candidates", [])[:2],
            "javascript_scope_signals": ch.get("javascript_scope_signals", [])[:2],
            "smell_type_signals": ch.get("smell_type_signals", [])[:3],
            "detection_technique_signals": ch.get("detection_technique_signals", [])[:2],
            "analysis_type_signals": ch.get("analysis_type_signals", [])[:2],
            "validation_type_signals": ch.get("validation_type_signals", [])[:2],
            "dataset_signals": ch.get("dataset_signals", [])[:2],
            "limitation_signals": ch.get("limitation_signals", [])[:2],
            "finding_signals": ch.get("finding_signals", [])[:2],
            "evidence_spans": ch.get("evidence_spans", [])[:2]
        })

    chunk_json = json.dumps(compact_chunks, ensure_ascii=False)

    intro = clean_text(sections.get("introduction", ""))[:2500]
    method = clean_text(sections.get("method", ""))[:2500]
    conclusion = clean_text(sections.get("conclusion", ""))[:2000]

    return f"""
Extract structured data for a systematic mapping study on tools and approaches for detecting code smells in JavaScript.

Rules:
1. Use only the evidence provided below.
2. Do not infer missing information.
3. If a field is absent or unclear, use "not_reported" or "not_clear".
4. smell_types must be a JSON list.
5. main_findings must be concise, maximum 3 sentences.
6. Return valid JSON only.

Study title:
{title}

Introduction (support only):
{intro}

Method (support only):
{method}

Conclusion (support only):
{conclusion}

Chunk evidence:
{chunk_json}

Return exactly this JSON:
{{
  "reference": "not_reported or extracted reference string",
  "tool_name": "not_reported",
  "artifact_type": "tool | approach | both | not_clear",
  "language_scope": "javascript | javascript_ecosystem | not_clear",
  "smell_types": ["..."],
  "detection_technique": "not_reported",
  "analysis_type": "static | dynamic | hybrid | not_clear",
  "validation_type": "toy_example | case_study | benchmark | comparative_evaluation | industrial_evaluation | not_clear",
  "dataset_or_projects": "not_reported",
  "limitations_reported": "not_reported",
  "main_findings": "not_reported",
  "evidence_tool_or_approach": "not_reported",
  "evidence_detection_technique": "not_reported",
  "evidence_validation": "not_reported",
  "evidence_limitations": "not_reported",
  "confidence": "HIGH | MEDIUM | LOW",
  "observations": "short note"
}}
""".strip()

## 10) Chamada ao modelo


In [ ]:
@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=2, max=30),
    retry=retry_if_exception_type((APITimeoutError, APIConnectionError, RateLimitError, APIError))
)
def run_model(prompt: str):
    response = client.responses.create(
        model=MODEL,
        input=[
            {
                "role": "developer",
                "content": (
                    "You are a rigorous structured-data extraction assistant for evidence synthesis. "
                    "Return valid JSON only. No markdown. No extra commentary."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    raw = response.output_text
    parsed = extract_json(raw)
    return parsed, raw


## 11) Checkpoint / retomada


In [ ]:
OUTPUT_COLUMNS = [
    "study_id",
    "pdf_file",
    "processing_status",     # DONE | ERROR
    "error_message",
    "title_extracted",
    "reference",
    "tool_name",
    "artifact_type",
    "language_scope",
    "smell_types",
    "detection_technique",
    "analysis_type",
    "validation_type",
    "dataset_or_projects",
    "limitations_reported",
    "main_findings",
    "evidence_tool_or_approach",
    "evidence_detection_technique",
    "evidence_validation",
    "evidence_limitations",
    "sections_detected",
    "chunk_count",
    "confidence",
    "observations",
    "text_file",
    "audit_json",
    "model",
    "prompt_version",
    "run_at",
    "review_status_human",
    "review_notes_human"
]

def load_checkpoint():
    csv_path = Path(OUTPUT_CSV)
    if csv_path.exists():
        df = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
        for col in OUTPUT_COLUMNS:
            if col not in df.columns:
                df[col] = ""
        df = df[OUTPUT_COLUMNS].copy()
        df = df.fillna("")
        return df

    return pd.DataFrame({col: pd.Series(dtype="object") for col in OUTPUT_COLUMNS})

def save_checkpoint(df):
    df = df[OUTPUT_COLUMNS].copy().fillna("")
    for col in df.columns:
        df[col] = df[col].astype(str)
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    df.to_excel(OUTPUT_XLSX, index=False)

def upsert_row(df, row_dict):
    # garante que tudo entre como texto
    normalized = {}
    for col in OUTPUT_COLUMNS:
        value = row_dict.get(col, "")
        if value is None:
            value = ""
        elif isinstance(value, list):
            value = "; ".join([str(v) for v in value])
        else:
            value = str(value)
        normalized[col] = value

    mask = df["pdf_file"] == normalized["pdf_file"]

    if mask.any():
        idx = df.index[mask][0]
        for k, v in normalized.items():
            df.at[idx, k] = v
    else:
        df.loc[len(df)] = normalized

    return df


## 12) Processamento de um PDF



In [ ]:
def process_pdf(pdf_path: Path, study_id: str):
    pdf_file = pdf_path.name
    title_extracted = extract_title_from_pdf(pdf_path)

    full_text = extract_full_text(pdf_path)

    txt_path = Path(TEXT_DIR) / f"{pdf_path.stem}.txt"
    txt_path.write_text(full_text, encoding="utf-8")

    sections, detected_sections = split_sections(full_text)
    chunks = chunk_text(full_text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)

    chunk_summaries = []
    raw_chunk_outputs = []

    for i, chunk in enumerate(chunks, start=1):
        prompt = build_chunk_prompt(title_extracted, i, chunk)
        parsed, raw = run_model(prompt)

        if parsed is None:
            parsed = {
                "tool_or_approach_candidates": [],
                "javascript_scope_signals": [],
                "smell_type_signals": [],
                "detection_technique_signals": [],
                "analysis_type_signals": [],
                "validation_type_signals": [],
                "dataset_signals": [],
                "limitation_signals": [],
                "finding_signals": [],
                "evidence_spans": [],
                "observations": "JSON_PARSE_ERROR"
            }

        chunk_summaries.append({
            "chunk_index": i,
            "tool_or_approach_candidates": parsed.get("tool_or_approach_candidates", []),
            "javascript_scope_signals": parsed.get("javascript_scope_signals", []),
            "smell_type_signals": parsed.get("smell_type_signals", []),
            "detection_technique_signals": parsed.get("detection_technique_signals", []),
            "analysis_type_signals": parsed.get("analysis_type_signals", []),
            "validation_type_signals": parsed.get("validation_type_signals", []),
            "dataset_signals": parsed.get("dataset_signals", []),
            "limitation_signals": parsed.get("limitation_signals", []),
            "finding_signals": parsed.get("finding_signals", []),
            "evidence_spans": parsed.get("evidence_spans", []),
            "observations": parsed.get("observations", "")
        })

        raw_chunk_outputs.append({
            "chunk_index": i,
            "raw_output": raw
        })

        time.sleep(REQUEST_PAUSE_SECONDS)

    consolidation_prompt = build_consolidation_prompt(
        title=title_extracted,
        sections=sections,
        chunk_summaries=chunk_summaries
    )

    parsed_final, raw_final = run_model(consolidation_prompt)

    if parsed_final is None:
        parsed_final = {
            "reference": "not_reported",
            "tool_name": "not_reported",
            "artifact_type": "not_clear",
            "language_scope": "not_clear",
            "smell_types": [],
            "detection_technique": "not_reported",
            "analysis_type": "not_clear",
            "validation_type": "not_clear",
            "dataset_or_projects": "not_reported",
            "limitations_reported": "not_reported",
            "main_findings": "not_reported",
            "evidence_tool_or_approach": "not_reported",
            "evidence_detection_technique": "not_reported",
            "evidence_validation": "not_reported",
            "evidence_limitations": "not_reported",
            "confidence": "LOW",
            "observations": "JSON_PARSE_ERROR"
        }

    audit_payload = {
        "study_id": study_id,
        "pdf_file": pdf_file,
        "title_extracted": title_extracted,
        "sections_detected": detected_sections,
        "chunk_count": len(chunks),
        "chunk_summaries": chunk_summaries,
        "raw_chunk_outputs": raw_chunk_outputs,
        "final_raw_output": raw_final
    }

    audit_file = Path(AUDIT_DIR) / f"{pdf_path.stem}.json"
    audit_file.write_text(json.dumps(audit_payload, ensure_ascii=False, indent=2), encoding="utf-8")

    return {
        "study_id": study_id,
        "pdf_file": pdf_file,
        "processing_status": "DONE",
        "error_message": "",
        "title_extracted": title_extracted,
        "reference": clean_text(parsed_final.get("reference", "not_reported")),
        "tool_name": clean_text(parsed_final.get("tool_name", "not_reported")),
        "artifact_type": clean_text(parsed_final.get("artifact_type", "not_clear")),
        "language_scope": clean_text(parsed_final.get("language_scope", "not_clear")),
        "smell_types": safe_join_list(parsed_final.get("smell_types", [])),
        "detection_technique": clean_text(parsed_final.get("detection_technique", "not_reported")),
        "analysis_type": clean_text(parsed_final.get("analysis_type", "not_clear")),
        "validation_type": clean_text(parsed_final.get("validation_type", "not_clear")),
        "dataset_or_projects": clean_text(parsed_final.get("dataset_or_projects", "not_reported")),
        "limitations_reported": clean_text(parsed_final.get("limitations_reported", "not_reported")),
        "main_findings": clean_text(parsed_final.get("main_findings", "not_reported")),
        "evidence_tool_or_approach": clean_text(parsed_final.get("evidence_tool_or_approach", "not_reported")),
        "evidence_detection_technique": clean_text(parsed_final.get("evidence_detection_technique", "not_reported")),
        "evidence_validation": clean_text(parsed_final.get("evidence_validation", "not_reported")),
        "evidence_limitations": clean_text(parsed_final.get("evidence_limitations", "not_reported")),
        "sections_detected": "; ".join(detected_sections) if detected_sections else "NONE",
        "chunk_count": len(chunks),
        "confidence": clean_text(parsed_final.get("confidence", "LOW")),
        "observations": clean_text(parsed_final.get("observations", "")),
        "text_file": str(txt_path),
        "audit_json": str(audit_file),
        "model": MODEL,
        "prompt_version": PROMPT_VERSION,
        "run_at": datetime.now(timezone.utc).isoformat(),
        "review_status_human": "",
        "review_notes_human": ""
    }

## 13) Loop principal com retomada


In [ ]:
ensure_dir(TEXT_DIR)
ensure_dir(AUDIT_DIR)

pdfs = list_pdfs(PDF_DIR)
if not pdfs:
    raise ValueError(f"Nenhum PDF encontrado em: {PDF_DIR}")

checkpoint_df = load_checkpoint()

done_set = set(
    checkpoint_df.loc[
        checkpoint_df["processing_status"].fillna("").eq("DONE"),
        "pdf_file"
    ].tolist()
)

error_set = set(
    checkpoint_df.loc[
        checkpoint_df["processing_status"].fillna("").eq("ERROR"),
        "pdf_file"
    ].tolist()
)

print(f"Já concluídos (DONE): {len(done_set)}")
print(f"Com erro anterior (ERROR): {len(error_set)}")

for idx, pdf_path in enumerate(tqdm(pdfs, desc="Extração estruturada com resume"), start=1):
    pdf_file = pdf_path.name

    if pdf_file in done_set:
        continue

    if (pdf_file in error_set) and (not REPROCESS_ERRORS):
        continue

    study_id = f"S{idx:02d}"

    # marca linha como RUNNING? aqui usamos apenas row transitória em memória
    try:
        result = process_pdf(pdf_path, study_id)
        checkpoint_df = upsert_row(checkpoint_df, result)
        done_set.add(pdf_file)
        if pdf_file in error_set:
            error_set.remove(pdf_file)

    except Exception as exc:
        err_msg = str(exc)
        error_row = {
            "study_id": study_id,
            "pdf_file": pdf_file,
            "processing_status": "ERROR",
            "error_message": err_msg[:1000],
            "title_extracted": "",
            "reference": "",
            "tool_name": "",
            "artifact_type": "",
            "language_scope": "",
            "smell_types": "",
            "detection_technique": "",
            "analysis_type": "",
            "validation_type": "",
            "dataset_or_projects": "",
            "limitations_reported": "",
            "main_findings": "",
            "evidence_tool_or_approach": "",
            "evidence_detection_technique": "",
            "evidence_validation": "",
            "evidence_limitations": "",
            "sections_detected": "",
            "chunk_count": "",
            "confidence": "",
            "observations": "PROCESSING_ERROR",
            "text_file": "",
            "audit_json": "",
            "model": MODEL,
            "prompt_version": PROMPT_VERSION,
            "run_at": datetime.now(timezone.utc).isoformat(),
            "review_status_human": "",
            "review_notes_human": ""
        }
        checkpoint_df = upsert_row(checkpoint_df, error_row)
        error_set.add(pdf_file)

        save_checkpoint(checkpoint_df)

        if STOP_ON_BILLING_ERROR and is_billing_error(exc):
            print("\nParando execução por provável erro de crédito/quota.")
            print(f"Último erro: {err_msg}")
            break

    save_checkpoint(checkpoint_df)
    time.sleep(REQUEST_PAUSE_SECONDS)

# salva final
save_checkpoint(checkpoint_df)

print("Concluído.")
print("CSV :", OUTPUT_CSV)
print("XLSX:", OUTPUT_XLSX)

print("\nResumo por status:")
print(checkpoint_df["processing_status"].value_counts(dropna=False))

display(checkpoint_df.head())

Já concluídos (DONE): 8
Com erro anterior (ERROR): 1


Extração estruturada com resume:   0%|          | 0/1 [00:00<?, ?it/s]

Concluído.
CSV : /content/drive/MyDrive/Colab Notebooks/TCC/extraction_matrix.csv
XLSX: /content/drive/MyDrive/Colab Notebooks/TCC/extraction_matrix.xlsx

Resumo por status:
processing_status
DONE    9
Name: count, dtype: int64


,study_id,pdf_file,processing_status,error_message,title_extracted,reference,tool_name,artifact_type,language_scope,smell_types,...,chunk_count,confidence,observations,text_file,audit_json,model,prompt_version,run_at,review_status_human,review_notes_human
0,S01,A large-scale empirical study of code smells i...,DONE,,A large-scale empirical study of code smells i...,A large-scale empirical study of code smells i...,not_reported,not_clear,javascript,Variable Re-assign; Assignment In Conditional ...,...,17,MEDIUM,Study focuses on fault-proneness and survival ...,/content/drive/MyDrive/Colab Notebooks/TCC/ext...,/content/drive/MyDrive/Colab Notebooks/TCC/ext...,gpt-4o,extraction_js_v2_resume,2026-04-06T01:32:11.222904+00:00,,
1,S02,An Empirical Study of Code Smells in JavaScrip...,DONE,,An Empirical Study of Code Smells in JavaScrip...,An Empirical Study of Code Smells in JavaScrip...,eslint,both,javascript,Variable Re-assign; Assignment In Conditional ...,...,10,HIGH,The study uses ESLint and statistical methods ...,/content/drive/MyDrive/Colab Notebooks/TCC/ext...,/content/drive/MyDrive/Colab Notebooks/TCC/ext...,gpt-4o,extraction_js_v2_resume,2026-04-06T00:38:32.189299+00:00,,
2,S03,Code Smell Detection Tool for Java Script Prog...,DONE,,Code Smell Detection Tool for Java Script Prog...,Code Smell Detection Tool for Java Script Prog...,not_reported,tool,javascript,Argument Count Mismatch; Argument Type Mismatc...,...,5,HIGH,TAJSlint and TAJSiint are key tools for detect...,/content/drive/MyDrive/Colab Notebooks/TCC/ext...,/content/drive/MyDrive/Colab Notebooks/TCC/ext...,gpt-4o,extraction_js_v2_resume,2026-04-06T00:38:58.581769+00:00,,
3,S04,Detecting code smells in React-based Web apps.pdf,DONE,,Information and Software Technology Detecting ...,Information and Software Technology Detecting ...,ReactSniffer,tool,javascript_ecosystem,Direct DOM Manipulation; Duplicated Component;...,...,11,HIGH,Evidence supports the existence of a detection...,/content/drive/MyDrive/Colab Notebooks/TCC/ext...,/content/drive/MyDrive/Colab Notebooks/TCC/ext...,gpt-4o,extraction_js_v2_resume,2026-04-06T00:40:03.815976+00:00,,
4,S05,DrAsync: Identifying and Visualizing Anti-Patt...,DONE,,DrAsync: Identifying and Visualizing Anti-Patt...,not_reported,DrAsync,tool,javascript,anti-patterns involving promises and async/awa...,...,10,HIGH,DrAsync efficiently detects and refactors Java...,/content/drive/MyDrive/Colab Notebooks/TCC/ext...,/content/drive/MyDrive/Colab Notebooks/TCC/ext...,gpt-4o,extraction_js_v2_resume,2026-04-06T00:41:12.827703+00:00,,
